# Day 63：毕业设计开发 Day 5：自动化测试、Ragas 评估报告、并发压测与 Token 成本优化（Jupyter Notebook 实战手册）

> **所属阶段**：第 6 阶段 · 第六阶段：毕业设计与就业冲刺 ｜ **第 9 周**：星辰智脑毕业设计全栈开发与上线  
> **GitLab 工单**：`STARMIND-063` ｜ **开发分支**：`feat/day63-ragas-loadtest-costopt`  
> **配套 HTML5 课件**：`courseware/day63/index.html` ｜ **独立版本快照**：`versions/day63/`

---

## 🎙️ 老陈开场旁白

早啊各位！毕业设计 Day 5，今天干三件简历上最加分的硬核事：Ragas 量化评估报告、异步并发压测、Token 成本优化！听老陈一句劝：答辩和面试时，别再说'我测试了几个问题觉得效果挺好'——这种话评委一听就皱眉。你要直接甩出一张数据表：Faithfulness 0.92、Context Recall 0.90、50 并发 P95 延迟 380ms、通过语义缓存 + 大小模型分级路由把月度 Token 成本直降 58%！今天咱们把这三套自动化评估与优化工具全写出来，跑出真实可复现的数据！

### 承上启下
- **昨日基线**：昨天我们打通了 FastAPI 全量 REST/SSE 接口并完成了前端流式联调，今天给系统做上线前的量化体检与降本增效。
- **明日预告**：明天（Day 64）进入毕业设计最后一天开发：Docker 容器化部署上线、健康检查巡检、演示脚本与交付文档。


In [ ]:
# 环境自检与路径初始化：确保可以无缝导入当日版本模块并支持异步协程运行
import os
import sys
from pathlib import Path

# 允许在 Jupyter 已有事件循环内安全调用 asyncio.run()
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'versions').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DAY_DIR = PROJECT_ROOT / 'versions' / 'day63'
for _p in (DAY_DIR, DAY_DIR / 'src', PROJECT_ROOT):
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
os.chdir(DAY_DIR)
__file__ = str(DAY_DIR / 'src' / 'notebook_entry.py')

print(f'✅ Python 版本: {sys.version.split()[0]}')
print(f'✅ 项目根目录: {PROJECT_ROOT}')
print(f'✅ 当日版本目录: {DAY_DIR}')


## 📋 企业需求背景与验收目标（STARMIND-063）

星辰智脑上线前需向技术评审委员会提交量化质量报告、高并发性能报告与 FinOps 成本优化方案，达到质量门禁后方可准予发布。

**今日迭代目标：**
- 实现 Ragas 风格四维评测器（Faithfulness / Answer Relevancy / Context Precision / Context Recall）并输出 Markdown 评测报告与发布门禁判定。
- 实现基于 `asyncio` 的并发压测引擎，统计 QPS、P50/P95/P99 延迟与成功率。
- 实现语义缓存（Semantic Cache）+ 难度分级路由（7B 本地 vs 旗舰云 API）成本优化器并量化降本比例。

**验收标准：**
- Ragas 评估器准确区分高忠实度回答与编造幻觉回答，并依据门禁阈值输出 PASS/FAIL。
- 并发压测器在 20+ 并发任务下正确计算 QPS 与分位延迟。
- 成本优化器在含重复 FAQ 与简单问答的流量集上实现 >= 40% 的 Token 费用节省。

---

## 📚 课堂核心技术笔记

### 1. 为什么看延迟必须看 P95/P99 而不是只看平均值（Avg）？

平均值极易被少数极快请求掩盖长尾卡顿。假设 95% 请求耗时 200ms，但 5% 请求超时 10 秒，平均耗时看起来只有 690ms，但实际上每 20 个员工就有 1 人体验崩溃。生产 SLA 永远盯紧 P95 和 P99。

> **⚠️ 老陈避坑**：压测样本数太少（如仅 5 条）时 P99 没有统计意义，生产压测至少跑 200~1000 条请求并剔除预热阶段。

### 2. Token 降本的第一原则：先砍无效调用，再换便宜模型

企业知识库问答往往符合二八定律：20% 的热门问题（考勤、报销、Wi-Fi、年假）占了 60% 以上的提问量。在网关前加一层语义缓存，不仅省钱，还能把热门问题延迟压到 10ms 以内。

> **⚠️ 老陈避坑**：涉及个人隐私数据（如'查我本月剩余年假'）的请求绝对不能进全局共享缓存，缓存键必须包含租户/用户隔离或仅对纯公共知识开启。


## 💻 实操步骤 1：步骤一：实现 Ragas 风格四维评测器与发布门禁报告生成器

- **目标文件**：`versions/day63/src/day63_ragas_evaluator.py`
- **老陈代码旁白**：手写轻量离线版 Ragas 四维评测器，对黄金测试集自动计算 Faithfulness、Relevancy、Precision、Recall 并输出 Markdown 验收报告。


In [ ]:
"""
Day 63 - 模块一：Ragas 风格四维 RAG 量化评估器与发布门禁报告
"""
from __future__ import annotations

import re
from dataclasses import dataclass
from typing import Dict, List, Set


def _tokens(text: str) -> Set[str]:
    low = text.lower()
    alnum = re.findall(r"[a-z0-9_]+", low)
    cjk = re.findall(r"[\u4e00-\u9fff]", low)
    return set(alnum + cjk)


def _overlap_ratio(source: Set[str], target: Set[str]) -> float:
    if not source:
        return 0.0
    return len(source & target) / len(source)


@dataclass
class EvalSample:
    question: str
    answer: str
    contexts: List[str]
    ground_truth: str


@dataclass
class RagasReport:
    faithfulness: float
    answer_relevancy: float
    context_precision: float
    context_recall: float
    passed: bool
    markdown: str


def evaluate_ragas(
    samples: List[EvalSample],
    min_faithfulness: float = 0.65,
    min_recall: float = 0.65,
) -> RagasReport:
    """计算四维均值并判定是否通过上线质量门禁"""
    f_list, rel_list, prec_list, rec_list = [], [], [], []
    for s in samples:
        ans_t = _tokens(s.answer)
        q_t = _tokens(s.question)
        gt_t = _tokens(s.ground_truth)
        ctx_all = set().union(*(_tokens(c) for c in s.contexts)) if s.contexts else set()

        # Faithfulness: 答案中多少比例的语义单元被上下文直接支撑
        faith = _overlap_ratio(ans_t, ctx_all)
        # Answer Relevancy: 答案覆盖提问与标准答案核心语义的综合程度
        relev = 0.5 * _overlap_ratio(q_t, ans_t) + 0.5 * _overlap_ratio(gt_t, ans_t)
        # Context Precision: 有多少比例的召回切片真正包含标准答案线索
        useful_ctx = sum(1 for c in s.contexts if _overlap_ratio(gt_t, _tokens(c)) >= 0.35)
        prec = useful_ctx / max(1, len(s.contexts))
        # Context Recall: 标准答案核心要点被上下文覆盖的比例
        rec = _overlap_ratio(gt_t, ctx_all)

        f_list.append(faith)
        rel_list.append(relev)
        prec_list.append(prec)
        rec_list.append(rec)

    n = max(1, len(samples))
    avg_f = round(sum(f_list) / n, 4)
    avg_rel = round(sum(rel_list) / n, 4)
    avg_prec = round(sum(prec_list) / n, 4)
    avg_rec = round(sum(rec_list) / n, 4)
    passed = (avg_f >= min_faithfulness) and (avg_rec >= min_recall)

    md = (
        "# 星辰智脑 Ragas 量化评估报告\n"
        f"- 样本数量: {len(samples)}\n"
        f"- Faithfulness (忠实度): {avg_f:.2%}\n"
        f"- Answer Relevancy (相关性): {avg_rel:.2%}\n"
        f"- Context Precision (上下文精度): {avg_prec:.2%}\n"
        f"- Context Recall (上下文召回): {avg_rec:.2%}\n"
        f"- 上线质量门禁: {'PASS' if passed else 'FAIL'}\n"
    )
    return RagasReport(avg_f, avg_rel, avg_prec, avg_rec, passed, md)


if __name__ == "__main__":
    golden = [
        EvalSample(
            question="一线城市差旅住宿标准是多少？",
            answer="一线城市差旅住宿上限650元/晚，需上传电子专票核销。",
            contexts=["一线城市出差住宿上限650元/晚，需在费用中心上传电子专票核销。"],
            ground_truth="一线城市住宿上限650元/晚，需上传电子专票。",
        ),
        EvalSample(
            question="ERR_4291 错误码什么意思？",
            answer="ERR_4291 表示部门日Token配额已触发硬熔断，请联系管理员提额。",
            contexts=["错误码ERR_4291表示部门日Token配额已触发硬熔断，请联系管理员提额。"],
            ground_truth="ERR_4291 代表部门日Token配额硬熔断，需联系管理员提额。",
        ),
    ]
    rep = evaluate_ragas(golden)
    print(rep.markdown)


### 🔍 步骤 1 代码深度解析

把 Ragas 核心四维评估拆解为纯 Python 确定性基线实现，可在无外网 LLM Judge 时快速在 CI 跑通回归门禁；生产环境可无缝接入 `ragas` 库配合评审大模型做语义级打分。

**预期运行输出参考：**
```text
# 星辰智脑 Ragas 量化评估报告
- 样本数量: 2
- Faithfulness (忠实度): 92.xx%
...
- 上线质量门禁: PASS
```


## 💻 实操步骤 2：步骤二：实现异步并发压测器与语义缓存+分级路由降本优化器

- **目标文件**：`versions/day63/src/day63_load_and_cost_opt.py`
- **老陈代码旁白**：压测高并发下的 P50/P95/P99 与 QPS，并用语义缓存 + 大小模型分级路由算出实打实的 Token 成本降幅。


In [ ]:
"""
Day 63 - 模块二：异步并发压测引擎 + 语义缓存与分级路由 Token 降本优化器
"""
from __future__ import annotations

import asyncio
import time
from dataclasses import dataclass
from typing import Awaitable, Callable, Dict, List, Tuple


@dataclass
class LoadTestResult:
    total_requests: int
    concurrency: int
    success_count: int
    error_rate: float
    qps: float
    p50_ms: float
    p95_ms: float
    p99_ms: float


def percentile(sorted_vals: List[float], p: float) -> float:
    if not sorted_vals:
        return 0.0
    idx = min(len(sorted_vals) - 1, int(round((p / 100.0) * (len(sorted_vals) - 1))))
    return round(sorted_vals[idx], 2)


async def run_load_test(
    worker_fn: Callable[[int], Awaitable[bool]],
    total_requests: int = 30,
    concurrency: int = 10,
) -> LoadTestResult:
    sem = asyncio.Semaphore(concurrency)
    latencies_ms: List[float] = []
    success = 0

    async def _one(i: int) -> None:
        nonlocal success
        async with sem:
            t0 = time.perf_counter()
            ok = await worker_fn(i)
            dt = (time.perf_counter() - t0) * 1000.0
            latencies_ms.append(dt)
            if ok:
                success += 1

    start = time.perf_counter()
    await asyncio.gather(*(_one(i) for i in range(total_requests)))
    elapsed = max(1e-6, time.perf_counter() - start)

    latencies_ms.sort()
    err_rate = round((total_requests - success) / max(1, total_requests), 4)
    return LoadTestResult(
        total_requests=total_requests,
        concurrency=concurrency,
        success_count=success,
        error_rate=err_rate,
        qps=round(total_requests / elapsed, 2),
        p50_ms=percentile(latencies_ms, 50),
        p95_ms=percentile(latencies_ms, 95),
        p99_ms=percentile(latencies_ms, 99),
    )


class TokenCostOptimizer:
    """L1 精确/规范化缓存 + L2 难度分级路由（7B 本地 vs 旗舰云 API）"""

    # 单次请求均价估算（元/千Token）：旗舰 0.020，本地 7B 0.002
    PRICE_PER_1K = {"flagship": 0.020, "local_7b": 0.002, "cache": 0.0}

    def __init__(self) -> None:
        self._cache: Dict[str, str] = {}

    @staticmethod
    def _norm(q: str) -> str:
        return "".join(q.strip().lower().split())

    @staticmethod
    def _is_complex(q: str) -> bool:
        return len(q) > 28 or any(k in q for k in ("多步", "对比", "架构", "合同审查", "代码重构"))

    def simulate_traffic(self, queries: List[Tuple[str, int]]) -> Dict[str, float]:
        """对比全量直连旗舰模型 vs 启用缓存+分级路由后的总成本"""
        baseline_cost = 0.0
        optimized_cost = 0.0
        cache_hits, local_hits, flagship_hits = 0, 0, 0

        for query, total_tokens in queries:
            baseline_cost += (total_tokens / 1000.0) * self.PRICE_PER_1K["flagship"]
            key = self._norm(query)
            if key in self._cache:
                cache_hits += 1
                continue
            self._cache[key] = "cached_ok"
            if self._is_complex(query):
                flagship_hits += 1
                optimized_cost += (total_tokens / 1000.0) * self.PRICE_PER_1K["flagship"]
            else:
                local_hits += 1
                optimized_cost += (total_tokens / 1000.0) * self.PRICE_PER_1K["local_7b"]

        saving_pct = round((baseline_cost - optimized_cost) / max(1e-9, baseline_cost) * 100.0, 2)
        return {
            "baseline_cny": round(baseline_cost, 4),
            "optimized_cny": round(optimized_cost, 4),
            "saving_pct": saving_pct,
            "cache_hits": cache_hits,
            "local_7b_hits": local_hits,
            "flagship_hits": flagship_hits,
        }


if __name__ == "__main__":
    async def mock_api(i: int) -> bool:
        await asyncio.sleep(0.005 if i % 10 != 0 else 0.015)
        return True

    lt = asyncio.run(run_load_test(mock_api, total_requests=30, concurrency=10))
    print("并发压测结果:", lt)

    opt = TokenCostOptimizer()
    traffic = [
        ("出差住宿标准是多少", 1200),
        ("出差住宿标准是多少", 1200),
        ("年假有几天", 900),
        ("年假有几天", 900),
        ("请多步对比三份采购合同审查风险并给出条款修改建议", 3200),
    ]
    print("Token 成本优化报告:", opt.simulate_traffic(traffic))


### 🔍 步骤 2 代码深度解析

`run_load_test` 利用协程信号量高保真模拟并发流量；`TokenCostOptimizer` 用直观的账单对比展示缓存命中 + 7B 分级路由如何把成本直降 50% 以上。

**预期运行输出参考：**
```text
并发压测结果: LoadTestResult(total_requests=30, concurrency=10, success_count=30, error_rate=0.0, qps=..., p50_ms=..., p95_ms=..., p99_ms=...)
Token 成本优化报告: {'baseline_cny': 0.148, 'optimized_cny': 0.0682, 'saving_pct': 53.92, ...}
```


## 🧪 当日自动化单元测试

在终端运行 `pytest versions/day63/tests/test_day63.py -v` 即可验收全部测试断言。下面展示完整测试脚本源码，供对照学习测试写法：

```python
"""Day 63 单元测试：Ragas 四维评测、异步并发压测与 Token 降本优化"""
import asyncio
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

from src.day63_ragas_evaluator import EvalSample, evaluate_ragas
from src.day63_load_and_cost_opt import TokenCostOptimizer, run_load_test


def test_ragas_passes_grounded_and_blocks_hallucination():
    good = [
        EvalSample(
            "一线城市住宿标准",
            "一线城市住宿上限650元/晚，需上传电子专票。",
            ["一线城市出差住宿上限650元/晚，需在费用中心上传电子专票核销。"],
            "一线城市住宿上限650元/晚，需电子专票。",
        )
    ]
    bad = [
        EvalSample(
            "一线城市住宿标准",
            "火星移民船票定价五百万星币免费送太空舱。",
            ["一线城市出差住宿上限650元/晚。"],
            "一线城市住宿上限650元/晚。",
        )
    ]
    assert evaluate_ragas(good).passed is True
    assert evaluate_ragas(bad).passed is False


def test_async_load_tester_metrics():
    async def mock_worker(i: int) -> bool:
        await asyncio.sleep(0.002)
        return i != 0

    res = asyncio.run(run_load_test(mock_worker, total_requests=20, concurrency=5))
    assert res.total_requests == 20
    assert res.success_count == 19
    assert res.error_rate == 0.05
    assert res.p95_ms >= res.p50_ms > 0
    assert res.qps > 0


def test_token_cost_optimizer_saves_over_40_percent():
    opt = TokenCostOptimizer()
    traffic = [
        ("住宿标准", 1000),
        ("住宿标准", 1000),
        ("报销流程", 1000),
        ("报销流程", 1000),
        ("请进行多步合同审查与架构对比分析", 2000),
    ]
    rep = opt.simulate_traffic(traffic)
    assert rep["cache_hits"] == 2
    assert rep["local_7b_hits"] == 2
    assert rep["flagship_hits"] == 1
    assert rep["saving_pct"] >= 40.0

```


## 📝 课后梯度实战作业与标准答案

建议先在下方新建空白单元格独立完成编码，运行通过后再核对参考答案。


### 作业 1（基础巩固）：基础巩固：为压测报告增加 SLA 达标判定函数

**题目要求：**
编写 `check_sla(result, max_p95_ms=500.0, max_error_rate=0.01) -> bool`，同时满足延迟和错误率要求才返回 `True`。

**思路提示：** 直接比较 `result.p95_ms <= max_p95_ms and result.error_rate <= max_error_rate`。


In [ ]:
from src.day63_load_and_cost_opt import LoadTestResult


def check_sla(res: LoadTestResult, max_p95_ms: float = 500.0, max_error_rate: float = 0.01) -> bool:
    return res.p95_ms <= max_p95_ms and res.error_rate <= max_error_rate


ok_res = LoadTestResult(100, 10, 100, 0.0, 220.0, 40.0, 180.0, 240.0)
bad_res = LoadTestResult(100, 10, 95, 0.05, 180.0, 50.0, 620.0, 900.0)
assert check_sla(ok_res) is True and check_sla(bad_res) is False
print("压测 SLA 门禁校验通过")


**答案解析与评分标准：**

把性能红线代码化后即可直接放进 GitLab CI 作为发布卡点。


### 作业 2（企业实战）：企业实战：给语义缓存增加敏感词/个人查询旁路（禁止缓存私人数据）

**题目要求：**
编写 `is_cacheable_query(query: str) -> bool`，若包含 `'我的'`、`'工资'`、`'绩效'`、`'报销单号'` 等个人隐私词，返回 `False` 跳过全局缓存。

**思路提示：** 检查关键词元组是否出现在 `query` 中。


In [ ]:
PRIVATE_HINTS = ("我的", "工资", "绩效", "报销单号", "身份证")


def is_cacheable_query(query: str) -> bool:
    return not any(h in query for h in PRIVATE_HINTS)


assert is_cacheable_query("差旅住宿标准是多少？") is True
assert is_cacheable_query("查一下我的报销单号进度") is False
print("隐私查询缓存旁路校验通过")


**答案解析与评分标准：**

公共制度可全局缓存，私人查询必须旁路直连，这是企业缓存设计的安全底线。


### 作业 3（架构进阶）：架构进阶：计算年化 FinOps 节省金额与 ROI 回本周期

**题目要求：**
给定每日请求量、优化前后单次成本与私有化 GPU 月租，计算月度净节省金额。

**思路提示：** 月度云端原成本 = 日请求 * 30 * 原单价；月度优化总成本 = 日请求 * 30 * 新单价 + GPU 月租。


In [ ]:
def calc_monthly_net_saving(daily_reqs: int, old_unit_cny: float, new_unit_cny: float, gpu_monthly_cny: float) -> float:
    old_total = daily_reqs * 30 * old_unit_cny
    new_total = daily_reqs * 30 * new_unit_cny + gpu_monthly_cny
    return round(old_total - new_total, 2)


saving = calc_monthly_net_saving(daily_reqs=50000, old_unit_cny=0.02, new_unit_cny=0.005, gpu_monthly_cny=4500.0)
assert saving == 18000.0
print(f"启用缓存+分级路由后月度净节省: {saving} 元")


**答案解析与评分标准：**

算清 GPU 固定成本与 API 边际成本的盈亏平衡点，是高级 AI 工程师向 CTO 汇报技术方案的必备能力。


---

## 🎯 老陈收尾复盘

硬核！Ragas 四维门禁、异步压测 P50/P95/P99 + QPS、语义缓存与分级路由降本对比报告全出炉了，这些真实量化数字直接写进明天的交付文档和后天的答辩 PPT！
